# Extraction and Embedding Script

The following is a script used to extract data from PDF files. The data was sourced from **TO BE CONTINUED**

In [9]:
!pip install docling

In [10]:
#-------------------------IMPORTS----------------------#
import csv
import json
import sqlite3
import time
import traceback

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sentence_transformers.util.tensor import normalize_embeddings
from transformers import AutoTokenizer

from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer

In [11]:
#-------------------------SQL CONFIG----------------------#

SCHEMA = """
CREATE TABLE IF NOT EXISTS documents (
    doc_id       TEXT PRIMARY KEY,   -- = CSV `index`
    filename     TEXT,
    source_url   TEXT,
    download_url TEXT,
    source_title TEXT,
    source_type  TEXT,
    status       TEXT NOT NULL,      -- 'done' | 'failed'
    error        TEXT,
    n_chunks     INTEGER DEFAULT 0,
    ingested_at  TEXT
);

CREATE TABLE IF NOT EXISTS chunks (
    chunk_id    INTEGER PRIMARY KEY AUTOINCREMENT,
    doc_id      TEXT NOT NULL REFERENCES documents(doc_id) ON DELETE CASCADE,
    chunk_index INTEGER NOT NULL,
    section     TEXT,
    page_start  INTEGER,
    page_end    INTEGER,
    text        TEXT NOT NULL,
    embed_text  TEXT NOT NULL,
    n_tokens    INTEGER,
    embedding   BLOB NOT NULL
);

CREATE INDEX IF NOT EXISTS idx_chunks_doc ON chunks(doc_id);

CREATE VIEW IF NOT EXISTS chunk_citations AS
SELECT c.chunk_id, c.doc_id, c.chunk_index, c.section, c.page_start, c.page_end, c.text,
       d.filename, d.source_url, d.download_url, d.source_title, d.source_type
FROM chunks c JOIN documents d ON d.doc_id = c.doc_id;

"""

In [12]:
#-------------------------DATABASE CREATION----------------------#

def init_db(path:  str) -> sqlite3.Connection:
  conn = sqlite3.connect(path)
  conn.execute("PRAGMA foreign_keys = ON;")   # needed for ON DELETE CASCADE
  conn.execute("PRAGMA journal_mode = WAL;")  # safer/faster writes
  conn.executescript(SCHEMA)
  conn.commit()
  return conn

In [13]:
#-------------------------EXTRACTION----------------------#
def build_chunker(tokenizer_name: str) -> HybridChunker:
  """HybridChunker that counts tokens with the SAME tokenizer as the embedder."""
  tokenizer = HuggingFaceTokenizer(
      tokenizer=AutoTokenizer.from_pretrained(tokenizer_name),
      max_tokens=MAX_CHUNK_TOKENS,
  )
  return HybridChunker(tokenizer=tokenizer, merge_peers=True)


def page_range(chunk) -> tuple[int | None, int | None]:
    """Collect page numbers of every item in the chunk."""
    pages = []
    for item in chunk.meta.doc_items:
        for prov in getattr(item, "prov", []) or []:
            pages.append(prov.page_no)
    return (min(pages), max(pages)) if pages else (None, None)


def extract_chunks(converter: DocumentConverter, chunker: HybridChunker, filepath: str):
  """Convert one document and return a list of chunk dicts"""
  # Convert from filepath
  result = converter.convert(filepath)
  converted_doc = result.document

  chunks = []
  # dl_doc is the document to chunk
  for i, chunk in enumerate(chunker.chunk(dl_doc=converted_doc)):
    start, end = page_range(chunk)
    headings = chunk.meta.headings or []
    chunks.append({
        "chunk_index": i,
        "section": " > ".join(headings) if headings else None,
        "page_start": start,
        "page_end": end,
        "text": chunk.text,
        # contextualize() prepends the headings to the text -> better embeddings
        "extracted_text": chunker.contextualize(chunk=chunk),
    })
  return chunks

In [14]:
#-------------------------EMBEDDING----------------------#
def load_embedder() -> SentenceTransformer:
  device = "cuda" if torch.cuda.is_available() else "cpu"
  model = SentenceTransformer(
      EMBED_MODEL,
      device=device,
      model_kwargs={"torch_dtype": torch.float16},
  )
  model.max_seq_length = MAX_CHUNK_TOKENS + 64    # headroom for the heading prefix
  return model

def embed_texts(model: SentenceTransformer, texts: list[str]) ->np.ndarray:
  """Embed without the query instruction prefix.
  Returns float32 array of shape (n, 1024), L2-normalised"""
  embedding_vectors = model.encode(
      inputs=texts,
      batch_size=EMBED_BATCH_SIZE,
      normalize_embeddings=True,
      convert_to_numpy=True,
  )
  return embedding_vectors.astype(np.float32)

In [15]:
#-------------------------SQL OPERATIONS----------------------#
# SQL database metadata fields
METADATA_FIELDS = ("doc_id", "filename", "source_url", "download_url", "source_title", "source_type")

def store_document(conn, document, chunks, vectors):
  """Write the document row and all its chunks in a transaction."""
  with conn:
    # Overwrite previous instance
    conn.execute("DELETE FROM documents WHERE doc_id = ?", (document["doc_id"],))
    # Add status, n_chunks, ingested_at to the metadata of the document
    conn.execute(
            f"""INSERT INTO documents ({", ".join(METADATA_FIELDS)}, status, n_chunks, ingested_at)
                VALUES (?, ?, ?, ?, ?, ?, 'done', ?, datetime('now'))""",
            (*[document[k] for k in METADATA_FIELDS], len(chunks)),
        )

    # Write many entries at once, many chunks
    conn.executemany(
            """INSERT INTO chunks
               (doc_id, chunk_index, section, page_start, page_end,
                text, embed_text, n_tokens, embedding)
               VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)""",
            [
                (document["doc_id"], chunk["chunk_index"], chunk["section"], chunk["page_start"],
                 chunk["page_end"], chunk["text"], chunk["extracted_text"], chunk["n_tokens"], vector.tobytes())
                for chunk, vector in zip(chunks, vectors)
            ],
        )


def mark_failed(conn, doc, error):
  """"Mark the document extraction as failed in the sql database."""
  with conn:
      conn.execute("DELETE FROM documents WHERE doc_id = ?", (doc["doc_id"],))
      conn.execute(
          f"""INSERT INTO documents ({", ".join(METADATA_FIELDS)}, status, error, ingested_at)
              VALUES (?, ?, ?, ?, ?, ?, 'failed', ?, datetime('now'))""",
          (*[doc[k] for k in METADATA_FIELDS], error),
      )

def already_done(conn: sqlite3.Connection, doc_id: str) -> bool:
    row = conn.execute(
        "SELECT status FROM documents WHERE doc_id = ?", (doc_id,)
    ).fetchone()
    # status has to be done and row cannot be empty to return true
    return row is not None and row[0] == "done"


def get_citations(conn, chunk_ids: list[int]) -> dict[int, dict]:
  """Return {chunk_id: citation dict} for the chunks listed by the retrieval algorithm."""
  conn.row_factory = sqlite3.Row
  # Placeholders ? are added for each chunk
  # Citation chunk added for each chunk_id
  marks = ",".join("?" * len(chunk_ids))
  rows = conn.execute(
      f"SELECT * FROM chunk_citations WHERE chunk_id IN ({marks})", chunk_ids
  ).fetchall()
  conn.row_factory = None
  return {r["chunk_id"]: dict(r) for r in rows}

In [21]:
import os
import zipfile


CSV_PATH ="/content/drive/MyDrive/RAG Documents/document_metadata.csv"
DOCS_PATH = "/content/drive/MyDrive/RAG Documents/rag_documents"
DB_PATH = "/content/drive/MyDrive/RAG Documents/student_assistant_rag_pt3.db"
EMBED_MODEL = "Qwen/Qwen3-Embedding-0.6B"
INDEX_TO_RESUME = 51
INDEX_TO_STOP = 63
MAX_CHUNK_TOKENS = 512
EMBED_BATCH_SIZE = 32

def main():
  conn = init_db(DB_PATH)
  converter = DocumentConverter()
  chunker = build_chunker(EMBED_MODEL)
  embedder = load_embedder()
  token_maker = chunker.tokenizer

  with open(CSV_PATH, newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

    # Index extracted files ONCE
    file_lookup = {}
    for root, dirs, files in os.walk(DOCS_PATH):
        for filename in files:
          file_lookup[filename] = os.path.join(root, filename)

    # Check for filename within file_lookup
    for n, row in enumerate(rows, 1):

      if n < INDEX_TO_RESUME:
        continue

      if n == INDEX_TO_RESUME:
        print(f"Resuming from index {INDEX_TO_RESUME}")

      if n > INDEX_TO_STOP:
        continue

      # Map CSV columns to document fields: 'index' becomes doc_id
      doc = {
            "doc_id": row["index"].strip(),
            "filename": row["filename"].strip(),
            "source_url": row["source_url"].strip(),
            "download_url": row["download_url"].strip(),
            "source_title": row["source_title"].strip(),
            "source_type": row["source_type"].strip(),
            }

      if already_done(conn, doc["doc_id"]):
        print(f"[{n}/{len(rows)}] skip {doc['doc_id']} (already ingested)")
        # Skip if document is already stored
        continue

      # Find file, all files are PDF
      file_path = file_lookup.get(doc["filename"] + ".pdf")

      if file_path == None:
        print(f"index: {n}, file_path: {file_path}")

      t0 = time.time()

      try:
        chunks = extract_chunks(converter=converter, chunker=chunker, filepath=file_path)
        if not chunks:
          raise ValueError("Docling returned no chunks (empty or unreadable file)")

        for chunk in chunks:
          chunk["n_tokens"] = token_maker.count_tokens(chunk["extracted_text"])

        vectors = embed_texts(embedder, [chunk["extracted_text"] for chunk in chunks])
        store_document(conn, doc, chunks, vectors)
        print(f"[{n}/{len(rows)}] {doc['doc_id']} '{doc['filename']}': "
                    f"{len(chunks)} chunks in {time.time() - t0:.1f}s")
      except Exception as e:
        traceback.print_exc()
        mark_failed(conn, doc, str(e))
        print("ERROR: ")
        print(f"[{n}/{len(rows)}] {doc['doc_id']}: FAILED ({e})")


  conn.close()

In [22]:
if __name__ == "__main__":
  main()

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Resuming from index 51


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

... (20 outputs left out)

[WARNING] 2026-11-13 22:08:52,911 [RapidOCR] main.py:164: The text detection result is empty
[WARNING] 2026-11-13 22:09:00,452 [RapidOCR] main.py:164: The text detection result is empty
[WARNING] 2026-11-13 22:09:00,715 [RapidOCR] main.py:164: The text detection result is empty
[WARNING] 2026-11-13 22:09:02,625 [RapidOCR] main.py:164: The text detection result is empty


[62/76] 62 'Studying in Germany Practical Guide for International Students': 172 chunks in 22.3s


[WARNING] 2026-11-13 22:09:14,664 [RapidOCR] main.py:164: The text detection result is empty
[WARNING] 2026-11-13 22:09:16,239 [RapidOCR] main.py:164: The text detection result is empty
[WARNING] 2026-11-13 22:09:16,473 [RapidOCR] main.py:164: The text detection result is empty
[WARNING] 2026-11-13 22:09:17,211 [RapidOCR] main.py:164: The text detection result is empty
[WARNING] 2026-11-13 22:09:18,214 [RapidOCR] main.py:164: The text detection result is empty
[WARNING] 2026-11-13 22:09:18,606 [RapidOCR] main.py:164: The text detection result is empty


[63/76] 63 'Studying in Germany - Questions and Answers for Parents': 69 chunks in 21.1s


## Database Correction: Chunk ID Alignment

**EXPLAIN WHY**

In [4]:

import sqlite3
import os

databases = [
    "/content/drive/MyDrive/RAG Documents/student_assistant_rag_pt1.db",
    "/content/drive/MyDrive/RAG Documents/student_assistant_rag_pt2.db",
    "/content/drive/MyDrive/RAG Documents/student_assistant_rag_pt3.db"
]

next_id = 1

for db in databases:

    print(f"\nProcessing {db}")

    if not os.path.exists(db):
        raise FileNotFoundError(db)

    conn = sqlite3.connect(db)
    cursor = conn.cursor()

    try:
        cursor.execute("BEGIN IMMEDIATE")

        # Read all existing chunk IDs
        cursor.execute("""
            SELECT chunk_id
            FROM chunks
            ORDER BY chunk_id
        """)

        old_ids = [row[0] for row in cursor.fetchall()]

        if not old_ids:
            print("No chunks found.")
            conn.commit()
            continue

        # First move all IDs to unique negative values
        # This prevents primary key collisions.
        temp_ids = {}

        for index, old_id in enumerate(old_ids):
            temp_id = -(index + 1)
            temp_ids[old_id] = temp_id

            cursor.execute("""
                UPDATE chunks
                SET chunk_id = ?
                WHERE chunk_id = ?
            """, (temp_id, old_id))

        # Assign globally sequential IDs
        for index, old_id in enumerate(old_ids):
            new_id = next_id + index
            temp_id = temp_ids[old_id]

            cursor.execute("""
                UPDATE chunks
                SET chunk_id = ?
                WHERE chunk_id = ?
            """, (new_id, temp_id))

        conn.commit()

        # Verify results
        cursor.execute("""
            SELECT MIN(chunk_id), MAX(chunk_id), COUNT(*)
            FROM chunks
        """)

        minimum, maximum, count = cursor.fetchone()

        print(f"Total chunks: {count}")
        print(f"Minimum ID: {minimum}")
        print(f"Maximum ID: {maximum}")

        next_id = maximum + 1

    except Exception as e:
        conn.rollback()
        print(f"ERROR: {e}")
        raise

    finally:
        conn.close()

print("\nAll databases processed.")
print(f"Total sequential chunk IDs: 1 to {next_id - 1}")



Processing /content/drive/MyDrive/RAG Documents/student_assistant_rag_pt1.db
Total chunks: 2756
Minimum ID: 1
Maximum ID: 2756

Processing /content/drive/MyDrive/RAG Documents/student_assistant_rag_pt2.db
Total chunks: 242
Minimum ID: 2757
Maximum ID: 2998

Processing /content/drive/MyDrive/RAG Documents/student_assistant_rag_pt3.db
Total chunks: 721
Minimum ID: 2999
Maximum ID: 3719

All databases processed.
Total sequential chunk IDs: 1 to 3719


## Database Concatenation

**EXPLAIN WHY**

In [6]:

import sqlite3
import os

# Input databases
databases = [
    "/content/drive/MyDrive/RAG Documents/student_assistant_rag_pt1.db",
    "/content/drive/MyDrive/RAG Documents/student_assistant_rag_pt2.db",
    "/content/drive/MyDrive/RAG Documents/student_assistant_rag_pt3.db"
]

# Output database
output_db = "student_assistant_rag_0_6B.db"

# Check input files
for db in databases:
    if not os.path.exists(db):
        raise FileNotFoundError(f"Database not found: {db}")

# Remove existing output database
if os.path.exists(output_db):
    os.remove(output_db)

# Connect to output database
conn = sqlite3.connect(output_db)
cursor = conn.cursor()

try:
    # Attach source databases
    for i, db in enumerate(databases):
        cursor.execute(
            f"ATTACH DATABASE ? AS source{i}",
            (os.path.abspath(db),)
        )

    # Copy tables from source databases
    for i, db in enumerate(databases):

        cursor.execute(f"""
            SELECT name, sql
            FROM source{i}.sqlite_master
            WHERE type = 'table'
            AND name NOT LIKE 'sqlite_%'
        """)

        tables = cursor.fetchall()

        for table_name, create_sql in tables:

            # Check whether table already exists
            cursor.execute("""
                SELECT name
                FROM main.sqlite_master
                WHERE type = 'table' AND name = ?
            """, (table_name,))

            exists = cursor.fetchone()

            if not exists:

                # Create table in destination
                cursor.execute(create_sql)

            # Copy records
            cursor.execute(f"""
                INSERT INTO main."{table_name}"
                SELECT * FROM source{i}."{table_name}"
            """)

            print(f"Copied {table_name} from {db}")

    # Create index on chunks
    cursor.execute("""
        CREATE INDEX IF NOT EXISTS idx_chunks_doc
        ON chunks(doc_id)
    """)

    print("Created index: idx_chunks_doc")

    # Create citations view
    cursor.execute("""
        CREATE VIEW IF NOT EXISTS chunk_citations AS
        SELECT
            c.chunk_id,
            c.doc_id,
            c.chunk_index,
            c.section,
            c.page_start,
            c.page_end,
            c.text,
            d.filename,
            d.source_url,
            d.download_url,
            d.source_title,
            d.source_type
        FROM chunks c
        JOIN documents d ON d.doc_id = c.doc_id
    """)

    print("Created view: chunk_citations")

    # Commit changes
    conn.commit()

    print(f"\nSuccessfully created {output_db}")

    # Verify tables and views
    cursor.execute("""
        SELECT name, type
        FROM main.sqlite_master
        WHERE type IN ('table', 'view', 'index')
        AND name NOT LIKE 'sqlite_%'
        ORDER BY type, name
    """)

    print("\nDatabase structure:")
    for name, object_type in cursor.fetchall():
        print(f"{object_type}: {name}")

except Exception as e:
    conn.rollback()
    print(f"Error: {e}")
    raise

finally:
    conn.close()


Copied documents from /content/drive/MyDrive/RAG Documents/student_assistant_rag_pt1.db
Copied chunks from /content/drive/MyDrive/RAG Documents/student_assistant_rag_pt1.db
Copied documents from /content/drive/MyDrive/RAG Documents/student_assistant_rag_pt2.db
Copied chunks from /content/drive/MyDrive/RAG Documents/student_assistant_rag_pt2.db
Copied documents from /content/drive/MyDrive/RAG Documents/student_assistant_rag_pt3.db
Copied chunks from /content/drive/MyDrive/RAG Documents/student_assistant_rag_pt3.db
Created index: idx_chunks_doc
Created view: chunk_citations

Successfully created student_assistant_rag_0_6B.db

Database structure:
index: idx_chunks_doc
table: chunks
table: documents
view: chunk_citations
